In [1]:
import requests
import polars as pl

seasons = [2020, 2021, 2022, 2023, 2024, 2025]
leagues = ["bl1", "bl2"]
all_matches = []

for season in seasons:
    for league in leagues:
        url = f"https://api.openligadb.de/getmatchdata/{league}/{season}"
        response = requests.get(url)
        if response.status_code != 200:
            continue
        data = response.json()
        if not data:
            continue

        for m in data:
            all_matches.append({
                "league":       league,
                "season":       season,
                "matchday":     m["group"]["groupOrderID"],
                "home":         m["team1"]["teamName"],
                "away":         m["team2"]["teamName"],
                "goals_home":   m["matchResults"][-1]["pointsTeam1"] if m["matchResults"] else None,
                "goals_away":   m["matchResults"][-1]["pointsTeam2"] if m["matchResults"] else None,
                "date":         m["matchDateTimeUTC"],
                "location_id":  m["location"]["locationID"]       if m["location"] else None,
                "city":         m["location"]["locationCity"]      if m["location"] else None,
                "stadium":      m["location"]["locationStadium"]   if m["location"] else None,
            })

df = pl.DataFrame(all_matches)

# Nur Heimspiele von Kiel
kiel_home = df.filter(pl.col("home").str.contains("Kiel"))

with pl.Config(tbl_rows=-1):
    print(kiel_home)
    kiel_home.write_csv("../../Datensaetze/kiel_heimspiele.csv")
    print(f"Gespeichert als kiel_heimspiele.csv ✓ ({len(kiel_home)} Spiele)")

shape: (102, 11)
┌────────┬────────┬──────────┬───────────────┬───┬──────────────────────┬─────────────┬──────┬───────────────────┐
│ league ┆ season ┆ matchday ┆ home          ┆ … ┆ date                 ┆ location_id ┆ city ┆ stadium           │
│ ---    ┆ ---    ┆ ---      ┆ ---           ┆   ┆ ---                  ┆ ---         ┆ ---  ┆ ---               │
│ str    ┆ i64    ┆ i64      ┆ str           ┆   ┆ str                  ┆ i64         ┆ str  ┆ str               │
╞════════╪════════╪══════════╪═══════════════╪═══╪══════════════════════╪═════════════╪══════╪═══════════════════╡
│ bl2    ┆ 2020   ┆ 1        ┆ Holstein Kiel ┆ … ┆ 2020-09-20T11:30:00Z ┆ null        ┆ null ┆ null              │
│ bl2    ┆ 2020   ┆ 3        ┆ Holstein Kiel ┆ … ┆ 2020-10-04T11:30:00Z ┆ null        ┆ null ┆ null              │
│ bl2    ┆ 2020   ┆ 5        ┆ Holstein Kiel ┆ … ┆ 2020-10-24T11:00:00Z ┆ null        ┆ null ┆ null              │
│ bl2    ┆ 2020   ┆ 7        ┆ Holstein Kiel ┆ … ┆ 2020-11-09T1